# Open and export a plasmid

Open the annotated pUC19 map, check it against its GenBank deposit, then export and reopen it in three formats. The notebook uses the same readers and writer as the desktop. [Import and export API](../importing_sequences.md) · [Plasmid sources](../../databases/plasmids/README.md)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "strandfolio" / "__init__.py").is_file())
sys.path.insert(0, str(root))

from strandfolio.io.formats import readSequence, readText, writeSequence

plasmids = root / "databases" / "plasmids"
maps = plasmids / "maps"

## Open pUC19

The map carries the four landmark labels used in the README. Compare its bases with accession M77789.2 and inspect the other annotated GenBank files that ship with the project. [pUC19 landmarks](../../databases/plasmids/README.md#puc19-map-labels)

In [ ]:
imported = readSequence(maps / "pUC19.gb")
puc19 = imported.records[0]
deposited = readSequence(plasmids / "sources" / "M77789.2.gb").records[0]

assert deposited.sourceId == "M77789.2" and puc19.sequence == deposited.sequence
print(f"{puc19.name} matches {deposited.sourceId} across {len(puc19):,} bp.")
print("Features:", ", ".join(feature.label for feature in puc19.features))

In [ ]:
for path in sorted(maps.glob("*.gb")):
    record = readSequence(path).records[0]
    print(f"{path.stem:<26} {len(record):>5,} bp   {record.topology}   {len(record.features)} features")

## Export the same record three ways

GenBank preserves the annotated map. FASTA carries a header and sequence, and plain text carries only the letters. The callback shows the writer’s fidelity report and accepts those differences for these example copies. Each run gets a fresh directory.

In [ ]:
runId = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = root / "docs/notebooks/_outputs/I09" / runId
output.mkdir(parents=True)

def acceptExampleExport(report):
    print(report.destinationPath.name, *report.warnings, *report.serializationDetails, sep="\n  ")
    return True

for suffix in ("gb", "fasta", "txt"):
    writeSequence(puc19, output / f"pUC19.{suffix}", confirm=acceptExampleExport)
print(output.relative_to(root))

## Reopen the exported files

The GenBank and FASTA readers recognize the file content. For headerless text, supply the molecule type explicitly. Compare the topology and feature count shown for each format.

In [ ]:
genbank = readSequence(output / "pUC19.gb")
fasta = readSequence(output / "pUC19.fasta", moleculeType="DNA")
text = readText(output / "pUC19.txt", moleculeType="DNA")

for result in (genbank, fasta, text):
    record = result.records[0]
    assert record.sequence == puc19.sequence
    print(result.formatName, record.topology, len(record.features), "features")
assert imported.captureBytes == (maps / "pUC19.gb").read_bytes()

## Open a SnapGene file

The same reader opens the bundled 40-base circular example. Its result includes the original captured bytes and warnings describing what the parser could recover. [SnapGene example](../../strandfolio/data/examples/README.md)

In [ ]:
snapgene = readSequence(root / "strandfolio/data/examples/synthetic_circular.dna")
example = snapgene.records[0]
print(f"{example.name}: {len(example)} bp, {example.topology}, {len(example.features)} features")
print(*snapgene.warnings, sep="\n")